In [1]:
from getpass import getpass
import pandas as pd
from sqlalchemy import URL, create_engine, text

engine = create_engine(URL.create(
    "postgresql+psycopg",
    host="localhost", port=5432,
    username="postgres",
    database="postgres",
    password=getpass("PostgreSQL 비밀번호: "),
))

pd.read_sql_query("SELECT current_database(), current_schema();", engine)

PostgreSQL 비밀번호:  ········


,current_database,current_schema
0,postgres,public


In [31]:
with engine.begin() as conn:
    query = text(
            "SELECT * FROM public.menu_items"
        )
    
    display(pd.read_sql(query, conn))

,menu_item_id,item_name,category,price
0,101,Hamburger,American,12.95
1,102,Cheeseburger,American,13.95
2,103,Hot Dog,American,9.00
3,104,Veggie Burger,American,10.50
4,105,Mac & Cheese,American,7.00
5,106,French Fries,American,7.00
6,107,Orange Chicken,Asian,16.50
7,108,Tofu Pad Thai,Asian,14.50
8,109,Korean Beef Bowl,Asian,17.95
9,110,Pork Ramen,Asian,17.95


In [5]:
with engine.connect() as conn:
    df = pd.read_sql(text("SELECT * FROM public.menu_items"), conn)

display(df.describe())

,menu_item_id,price
count,32.000000,32.000000
mean,116.500000,13.285937
std,9.380832,3.858071
min,101.000000,5.000000
25%,108.750000,10.500000
50%,116.500000,13.950000
75%,124.250000,15.750000
max,132.000000,19.950000


In [6]:
with engine.connect() as conn:
    df = pd.read_sql(text("SELECT * FROM public.order_details"), conn)

display(df.describe())

,order_details_id,order_id,item_id
count,12234.000000,12234.000000,12097.000000
mean,6117.500000,2691.927415,115.202282
std,3531.795931,1546.026261,9.387580
min,1.000000,1.000000,101.000000
25%,3059.250000,1351.250000,107.000000
50%,6117.500000,2710.000000,114.000000
75%,9175.750000,4020.000000,123.000000
max,12234.000000,5370.000000,132.000000


In [37]:
# objective 01

with engine.begin() as conn:
    obj_01_02_max = text("select * from public.menu_items order by price desc limit 1;")
    obj_01_02_min = text("select * from public.menu_items order by price limit 1; ")
    
    for i in [obj_01_02_max, obj_01_02_min]:
        display(pd.read_sql(i, conn))

    obj_01_03_cnt = text("select count(*) from public.menu_items where category='Italian';")
    obj_01_03_max = text("select item_name, price from public.menu_items where category='Italian' order by price desc limit 1;")
    obj_01_03_min = text("select item_name, price from public.menu_items where category='Italian' order by price limit 1;")

    for i in [obj_01_03_cnt, obj_01_03_max, obj_01_03_min]:
        display(pd.read_sql(i, conn))

    obj_01_04_cnt = text("select category, count(item_name) as category_cnt from public.menu_items group by category ;")
    obj_01_04_avg = text("select category, count(item_name) as category_cnt, AVG(price) as category_avg from public.menu_items group by category;")

    for i in [obj_01_04_cnt, obj_01_04_avg]:
        display(pd.read_sql(i, conn))


,menu_item_id,item_name,category,price
0,130,Shrimp Scampi,Italian,19.95


,menu_item_id,item_name,category,price
0,113,Edamame,Asian,5.0


,count
0,9


,item_name,price
0,Shrimp Scampi,19.95


,item_name,price
0,Spaghetti,14.5


,category,category_cnt
0,American,6
1,Mexican,9
2,Asian,8
3,Italian,9


,category,category_cnt,category_avg
0,American,6,10.066667
1,Mexican,9,11.800000
2,Asian,8,13.475000
3,Italian,9,16.750000


In [7]:
# objective 02

with engine.begin() as conn:
    obj_02_01 = text("select * from public.order_details;")

    display(pd.read_sql(obj_02_01, conn))

    obj_02_02_cnt = text("SELECT COUNT(*) FROM public.order_details")
    obj_02_02_itmcnt = text("SELECT item_id, COUNT(item_id) FROM public.order_details GROUP BY item_id;")

    for i in [obj_02_02_cnt, obj_02_02_itmcnt]:
        display(pd.read_sql(i, conn))

    obj_02_03_max = text("select order_id, count(item_id) as item_cnt from public.order_details group by order_id order by item_cnt desc limit 1;")

    display(pd.read_sql(obj_02_03_max, conn))

    obj_02_04 = text("select order_id, count(item_id) as item_cnt from public.order_details group by order_id having count(item_id) >= 12;")

    display(pd.read_sql(obj_02_04, conn))

,order_details_id,order_id,order_date,order_time,item_id
0,1,1,2023-01-01,11:38:36,109.0
1,2,2,2023-01-01,11:57:40,108.0
2,3,2,2023-01-01,11:57:40,124.0
3,4,2,2023-01-01,11:57:40,117.0
4,5,2,2023-01-01,11:57:40,129.0
...,...,...,...,...,...
12229,12230,5369,2023-03-31,22:05:04,109.0
12230,12231,5369,2023-03-31,22:05:04,129.0
12231,12232,5369,2023-03-31,22:05:04,120.0
12232,12233,5369,2023-03-31,22:05:04,122.0


,count
0,12234


,item_id,count
0,103.0,257
1,NaN,0
2,116.0,214
3,121.0,233
4,117.0,455
5,125.0,470
6,113.0,620
7,119.0,379
8,105.0,463
9,132.0,420


,order_id,item_cnt
0,2675,14


,order_id,item_cnt
0,2675,14
1,3583,13
2,2126,13
3,740,12
4,3654,12
5,2075,13
6,3473,14
7,3940,12
8,4732,12
9,144,12


In [2]:
# objective 03

with engine.begin() as conn:

    obj_03_01 = text("select * from public.menu_items mi join public.order_details od on mi.menu_item_id = od.item_id;")

    display(pd.read_sql(obj_03_01, conn))

    obj_03_02_max = text("select mi.menu_item_id, count(mi.menu_item_id) as menu_cnt, mi.category from public.menu_items mi join public.order_details od on mi.menu_item_id = od.item_id group by mi.menu_item_id order by menu_cnt desc limit 1;") 
    obj_03_02_min = text("select mi.menu_item_id, count(mi.menu_item_id) as menu_cnt, mi.category from public.menu_items mi join public.order_details od on mi.menu_item_id = od.item_id group by mi.menu_item_id order by menu_cnt limit 1;")

    for i in [obj_03_02_max, obj_03_02_min]:
        display(pd.read_sql(i, conn))

    obj_03_03 = text("select od.order_id , sum(price) as tot_price from public.menu_items mi join public.order_details od on mi.menu_item_id = od.item_id group by od.order_id order by tot_price desc limit 5;")

    display(pd.read_sql(obj_03_03, conn))

    obj_03_04 = text("select od.order_id, sum(price) as tot_price, ARRAY_AGG(mi.item_name) as orderd_menu from public.menu_items mi join public.order_details od on mi.menu_item_id = od.item_id group by od.order_id order by tot_price desc limit 5;")

    display(pd.read_sql(obj_03_04, conn))

,menu_item_id,item_name,category,price,order_details_id,order_id,order_date,order_time,item_id
0,109,Korean Beef Bowl,Asian,17.95,1,1,2023-01-01,11:38:36,109
1,108,Tofu Pad Thai,Asian,14.50,2,2,2023-01-01,11:57:40,108
2,124,Spaghetti,Italian,14.50,3,2,2023-01-01,11:57:40,124
3,117,Chicken Burrito,Mexican,12.95,4,2,2023-01-01,11:57:40,117
4,129,Mushroom Ravioli,Italian,15.50,5,2,2023-01-01,11:57:40,129
...,...,...,...,...,...,...,...,...,...
12092,109,Korean Beef Bowl,Asian,17.95,12230,5369,2023-03-31,22:05:04,109
12093,129,Mushroom Ravioli,Italian,15.50,12231,5369,2023-03-31,22:05:04,129
12094,120,Steak Torta,Mexican,13.95,12232,5369,2023-03-31,22:05:04,120
12095,122,Chips & Salsa,Mexican,7.00,12233,5369,2023-03-31,22:05:04,122


,menu_item_id,menu_cnt,category
0,101,622,American


,menu_item_id,menu_cnt,category
0,115,123,Mexican


,order_id,tot_price
0,440,192.15
1,2075,191.05
2,1957,190.10
3,330,189.70
4,2675,185.10


,order_id,tot_price,orderd_menu
0,440,192.15,"[Steak Tacos, Hot Dog, Spaghetti, Spaghetti & ..."
1,2075,191.05,"[Orange Chicken, Chicken Tacos, Cheeseburger, ..."
2,1957,190.10,"[Orange Chicken, Hot Dog, Veggie Burger, Fettu..."
3,330,189.70,"[Orange Chicken, Hot Dog, Tofu Pad Thai, Tofu ..."
4,2675,185.10,"[Hamburger, Cheeseburger, Tofu Pad Thai, Spagh..."
